<a href="https://colab.research.google.com/github/Maxstef/data-loves-ml-for-people-course/blob/main/notebooks/4_1_nlp_intro/0_7_bert_finetune_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 🔐 Fine-tuning BERT for Phishing URL Classification

In this notebook, we will **fine-tune** BERT (~110M parameters) to classify URLs as **Safe** or **Phishing**.
We'll go through the **full practical pipeline**: data → tokenization → training → evaluation → saving the model.

---

### 🧠 What is fine-tuning?

**Fine-tuning** means taking a model already pretrained on massive text (Wikipedia + Books) and **adapting it to your specific task** by training a small classification head and slightly adjusting the internal weights.

During pretraining, BERT learned language using two objectives:

* **Masked Language Modeling (MLM)** — predict missing words from both sides of context

  > “The weather is [MASK] today.” → *nice*

* **Next Sentence Prediction (NSP)** — understand relationships between sentences

Because of this, BERT already understands **syntax, context, and sentence structure**.
Fine-tuning teaches it how to apply that knowledge to **your labels** (Safe vs Phishing).

---

### 🧩 Why BERT is a great fit for URL classification

Although URLs are not natural sentences, they contain **meaningful patterns**:

* words: `login`, `secure`, `verify`, `account`
* structure: subdomains, paths, tokens, separators
* obfuscation tricks common in phishing

BERT’s **subword tokenization** and contextual understanding help it learn these patterns effectively.

---

### 🤔 Why fine-tune instead of calling an API LLM?

Fine-tuning a local BERT model is:

* ✅ **Free at inference time**
* ✅ **Private** (no data leaves your environment)
* ✅ **Fast** (milliseconds per prediction)
* ✅ **Easy to deploy** in APIs/backends
* ✅ **Fully controllable** (you own the model and data)

Use this approach when you need **privacy, speed, and independence** from external services.

---

### 🛠️ Best practices for BERT fine-tuning (classification)

* Use `AutoModelForSequenceClassification`
* Keep `label2id` / `id2label` consistent
* Use `DataCollatorWithPadding` for dynamic batching
* Track validation metrics and apply early stopping
* Save **model + tokenizer + labels** together for reuse

---

👉 Ready? Let’s install the libraries and start building the pipeline step by step.


In [1]:
!pip install transformers datasets evaluate torch --quiet

In [2]:
# !pip show evaluate

## 📦 Imports & Setup

We’ll use three core libraries from the Hugging Face ecosystem:

* datasets → ready-to-use NLP datasets
* transformers → pretrained BERT + training tools
* evaluate → standard metrics (accuracy, f1, etc.)


In [3]:
from datasets import load_dataset

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

import evaluate
import numpy as np
from transformers import DataCollatorWithPadding


---

## 📚 Loading a Dataset

Instead of preparing data manually, we can pull a phishing URL dataset directly from Hugging Face.

For this tutorial, use:

**`shawhin/phishing-site-classification`**

This dataset already contains:

* `url` — the input text
* `label` — 0 (safe) or 1 (phishing)
* predefined train / test splits

Perfect for practicing BERT fine-tuning without extra preprocessing.


In [4]:
dataset_dict = load_dataset(
    "shawhin/phishing-site-classification"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [5]:
dataset_dict

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 2100
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 450
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 450
    })
})

## 🤖 Load the Model and Tokenizer

At this stage, we load a pretrained BERT and the matching tokenizer from
**Transformers**.

* `AutoTokenizer.from_pretrained(...)` → automatically picks the correct tokenizer
* `AutoModelForSequenceClassification.from_pretrained(...)` → loads BERT **with a classification head on top**

We explicitly define:

* `num_labels=2` → **Safe** vs **Phishing**
* `id2label` / `label2id` → lets the model map numbers ↔ human-readable classes during training and inference (useful for logs, metrics, and predictions)

👉 Under the hood, this is **BERT + a small linear layer** that converts the 768-dim embedding into **2 class logits**.



In [ ]:
model_path = "google-bert/bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_path)

label2id = {"safe": 0, "phishing": 1}
id2label = {0: "safe", 1: "phishing"}

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
)

### ⚠️ About the warning when loading `BertForSequenceClassification`

You may see:

> *Some weights of BertForSequenceClassification were not initialized from the model checkpoint at `google-bert/bert-base-uncased` and are newly initialized: `['classifier.bias', 'classifier.weight']`*

This means:

> **The classification head was created from scratch and was not loaded from the pretrained checkpoint.**

**🔍 Why this happens**

* `bert-base-uncased` is a pretrained **language model** (MLM + NSP), not a classifier.
* `BertForSequenceClassification` adds an extra **linear layer (`classifier`)** on top of BERT.
* That layer **does not exist** in the original checkpoint, so **Hugging Face** initializes it with random weights.

**✅ Why this is expected (and correct)**

This is exactly how fine-tuning works:

1. Load pretrained BERT encoder (already understands language)
2. Add a small classification head
3. Train this head (and slightly adjust BERT) on your task

💡 Until you train the model, the classifier is random - so predictions will be meaningless.



---

### ❓ Can `bert-base-uncased` be used “as is” for classification?

**No — not directly.**

BERT base uncased is pretrained for **language understanding** (MLM + NSP), **not** for classifying text into labels like *“Safe”* / *“Not Safe”*.

You have two options:

1. **Fine-tune BERT on your dataset** → ✅ the standard and recommended approach.
2. **Use zero-shot classification** with a model already trained for entailment-style labeling.

Example:

```python
from transformers import pipeline

classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

result = classifier(
    "your input",
    candidate_labels=["Safe", "Not Safe"]
)

print(result)
```

This uses BART large MNLI, which was trained on Natural Language Inference and can generalize to unseen labels.

> 💡 Note: this is **not BERT**.
> It’s a model specifically trained to decide whether a sentence *entails* a label — which is why zero-shot works.


In [7]:
import torch

# model.eval() won't help - classificator not trained
model.eval()
inputs = tokenizer("test text", return_tensors="pt")
with torch.no_grad():
    logits = model(**inputs).logits
print(logits)

tensor([[-0.0975, -0.0584]])


We got random logits because classifier.weight and bias are not yet trained.

## 🧊 Preparing the model (Freezing most layers)

Before training, we **freeze the BERT encoder** and allow only the **final classification layers** to learn.

With BERT base uncased, the flow is:

```
tokens → BERT encoder → [CLS] embedding → classifier → label
```

We keep the encoder weights fixed and train only the **classifier head** (optionally the pooler). This turns BERT into a strong **feature extractor** while a small head learns your task.

**Why do this?**

* Faster training
* Works well for small datasets
* Reduces overfitting
* More stable optimization

Optionally, you can **unfreeze more layers** for full fine-tuning if needed.


In [8]:
# Show first and last 10 parameter entries
params = list(model.named_parameters())
layer_count = len(params)

for i, (name, param) in enumerate(params):
    if i < 10 or i >= layer_count - 10:
        print(f"{i:4d} | {name:60s} | requires_grad={param.requires_grad}")
    elif i == 10:
        print(" ...")

   0 | bert.embeddings.word_embeddings.weight                       | requires_grad=True
   1 | bert.embeddings.position_embeddings.weight                   | requires_grad=True
   2 | bert.embeddings.token_type_embeddings.weight                 | requires_grad=True
   3 | bert.embeddings.LayerNorm.weight                             | requires_grad=True
   4 | bert.embeddings.LayerNorm.bias                               | requires_grad=True
   5 | bert.encoder.layer.0.attention.self.query.weight             | requires_grad=True
   6 | bert.encoder.layer.0.attention.self.query.bias               | requires_grad=True
   7 | bert.encoder.layer.0.attention.self.key.weight               | requires_grad=True
   8 | bert.encoder.layer.0.attention.self.key.bias                 | requires_grad=True
   9 | bert.encoder.layer.0.attention.self.value.weight             | requires_grad=True
 ...
 191 | bert.encoder.layer.11.intermediate.dense.weight              | requires_grad=True
 192 | bert.enco

In [9]:
# Freeze whole BERT
for param in model.bert.parameters():
    param.requires_grad = False

# Unfreeze pooling layers
for param in model.bert.pooler.parameters():
    param.requires_grad = True

# Unfreeze classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

In [10]:
for i, (name, param) in enumerate(params):
    if i < 10 or i >= layer_count - 10:
        print(f"{i:4d} | {name:60s} | requires_grad={param.requires_grad}")
    elif i == 10:
        print(" ...")

   0 | bert.embeddings.word_embeddings.weight                       | requires_grad=False
   1 | bert.embeddings.position_embeddings.weight                   | requires_grad=False
   2 | bert.embeddings.token_type_embeddings.weight                 | requires_grad=False
   3 | bert.embeddings.LayerNorm.weight                             | requires_grad=False
   4 | bert.embeddings.LayerNorm.bias                               | requires_grad=False
   5 | bert.encoder.layer.0.attention.self.query.weight             | requires_grad=False
   6 | bert.encoder.layer.0.attention.self.query.bias               | requires_grad=False
   7 | bert.encoder.layer.0.attention.self.key.weight               | requires_grad=False
   8 | bert.encoder.layer.0.attention.self.key.bias                 | requires_grad=False
   9 | bert.encoder.layer.0.attention.self.value.weight             | requires_grad=False
 ...
 191 | bert.encoder.layer.11.intermediate.dense.weight              | requires_grad=False
 192 

## 🧾 Text Preprocessing

Before training, we need to convert raw text into token IDs that BERT can understand using BERT base uncased tokenizer.


### 🔤 Tokenization function

We define a preprocessing function that:

In [11]:
# define text preprecessing
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

### 📦 Apply to the full dataset

We apply tokenization to the entire dataset using batch processing (faster and more efficient):


In [12]:
# tokenize full dataset
tokenized_data = dataset_dict.map(preprocess_function, batched=True)


---

### 📏 Dynamic Padding with Data Collator

Texts in a batch often have **different lengths**, so we cannot stack them directly into tensors.

To solve this, we use:

`DataCollatorWithPadding`

---

**🧠 What it does**

* Finds the longest sequence in each batch
* Pads all other sequences to the same length
* Creates properly shaped tensors for training

---

**⚙️ Why it is important**

Without dynamic padding:

* you would waste memory (fixed max-length padding everywhere)
* or get shape mismatch errors during batching

With it:

* padding is applied **only when needed per batch**
* training becomes more efficient and flexible


In [13]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

## 📊 Evaluation Metrics

To evaluate our fine-tuned BERT base uncased classifier, we use standard classification metrics from the Evaluate library.

We measure:

* **Accuracy** → how many predictions are correct
* **ROC AUC** → how well the model separates *Safe vs Phishing*


In [14]:
# Load metrics
accuracy = evaluate.load("accuracy")
auc_score = evaluate.load("roc_auc")

### 🧠 Metric function for Trainer

This function is called automatically during evaluation.

It:

* converts model logits → probabilities
* extracts positive class scores (phishing = 1)
* computes AUC
* computes accuracy

👉 AUC is especially important for **imbalanced datasets**, which is common in phishing detection.

In [15]:
def compute_metrics(eval_pred):
    # Get predictions and real labels
    predictions, labels = eval_pred

    # apply softmax to obtain probabilities
    probabilities = np.exp(predictions) / np.exp(predictions).sum(-1, keepdims=True)

    # take the positive class probabilities to calculate the AUC
    positive_class_probs = probabilities[:, 1]

    # calculate AUC
    auc = np.round(auc_score.compute(prediction_scores=positive_class_probs, references=labels)['roc_auc'], 3)

    # get the predicted classes (most likely)
    predicted_classes = np.argmax(predictions, axis=1)

    # calculate accuracy
    acc = np.round(accuracy.compute(predictions=predicted_classes, references=labels)['accuracy'], 3)

    return {"Accuracy": acc, "AUC": auc}

## 🏋️ Train the Model

Now we define the training configuration and fine-tune our BERT base uncased using the Hugging Face training pipeline.

We use the **Trainer API** from Transformers, which handles:

* training loop
* evaluation
* logging
* checkpointing


### ⚙️ Hyperparameters

* **learning_rate** → how fast the model updates weights
* **batch_size** → number of samples per step
* **epochs** → how many times the model sees the dataset

In [16]:
# hyperparameters
lr = 2e-4
batch_size = 8
num_epochs = 10

### 🧠 Training configuration

👉 This setup ensures:

* evaluation after each epoch
* best model automatically saved
* training logs tracked per epoch

In [17]:
training_args = TrainingArguments(
    output_dir="bert-phishing-classifier",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

### 🏁 Start training

#### ⚠️ Important note (performance)

Training BERT base uncased on CPU is **very slow**, because:

* ~110 million parameters
* transformer attention is computationally expensive

👉 Recommended setup:

* Google Colab GPU (T4 / A100)
* or local CUDA-enabled GPU

In [18]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Use GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not availabel. Use CPU.")

Use GPU: Tesla T4


In [ ]:
%%time
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    # tokenizer=tokenizer, # deprecated in transformers versions (≥ 4.44)
    processing_class=tokenizer, # should be used instead in transformers versions (≥ 4.44)
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

### 🧪 Interpreting the First Training Run (Frozen Encoder + Trainable Pooler & Head)

In the first experiment with **BERT base uncased**, setup was:

* ❄️ Freeze **embeddings** and all **12 encoder layers**
* ✅ Train **`bert.pooler`**
* ✅ Train **`classifier` head**

So the trainable path looked like:

```
[CLS] from frozen BERT → Pooler (trainable) → Classifier (trainable)
```

---

**📊 What the metrics showed**

| Epoch | Val Loss |  Accuracy |       AUC |
| ----: | -------: | --------: | --------: |
|     1 |    0.366 |     0.858 |     0.926 |
|     6 |    0.309 |     0.887 |     0.942 |
|     9 |    0.303 | **0.893** | **0.945** |

We reached **~89% accuracy** and **0.945 AUC** **without updating a single Transformer layer**.

---

**🧠 What was actually learned**

By unfreezing **pooler**, we allowed the model to:

* Re-learn **how to interpret the `[CLS]` sentence representation** for *phishing detection*
* Adapt the **sentence-level meaning**, while keeping BERT’s language knowledge intact

This is **more than “classifier-only” training**.
We effectively fine-tuned the **sentence representation layer** that sits between BERT and the classifier.

---

**✅ Takeaway**

* The frozen encoder already produced very strong contextual features for URLs/text.
* We only needed to retrain:

  * how `[CLS]` is transformed (**pooler**)
  * how that maps to labels (**classifier**)
* This is a form of **light fine-tuning**:

  * much faster than full fine-tuning
  * surprisingly strong performance

This result naturally leads to the next question:

👉 *What happens if we also unfreeze the last encoder layers?*


### 🧪 Evaluate the Quality on the Test Set

After training, we evaluate the model on a **previously unseen** test split to measure how well it generalizes.


In [20]:
# generate prediction
predictions = trainer.predict(tokenized_data["test"])

# get logits and labels
logits = predictions.predictions
labels = predictions.label_ids

# use created earlier compute_metrics
metrics = compute_metrics((logits, labels))
print(metrics)

{'Accuracy': np.float64(0.873), 'AUC': np.float64(0.951)}


### Snapshot stage 1 model

In [21]:
import copy
stage1_model = copy.deepcopy(model)
stage1_tokenizer = copy.deepcopy(tokenizer)

In [ ]:
trainer.model = stage1_model
trainer.save_model("bert-stage1")
trainer.push_to_hub("maksymstefanko/bert-phishing-stage1")

https://huggingface.co/maksymstefanko/bert-phishing-classifier

## 🚀 Stage 2 Fine-Tuning - Unfreezing the Top Transformer Layers

In Stage 1, we trained only:

```
[CLS] → pooler → classifier
```

The entire Transformer encoder stayed frozen.

Now we allow **the top of the encoder** to adapt to the phishing task.

In [23]:
for name, param in model.base_model.named_parameters():
    if "encoder.layer.10" in name or "encoder.layer.11" in name:
        param.requires_grad = True

This unfreezes the **last 2 encoder layers** (layers 11 and 12 out of 12).

**🧠 Why the last layers?**

In **BERT base uncased**:

* Lower layers → generic language patterns (syntax, grammar)
* Middle layers → phrase semantics
* Top layers → **task-specific meaning**

By unfreezing only the top layers, we let BERT slightly **reshape its understanding of URLs** without destroying its pretrained language knowledge.


In [24]:
# new training arguments
training_args_stage2 = TrainingArguments(
    output_dir="bert-phishing-classifier-stage2",
    learning_rate=2e-5,          # 10x smaller
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,          # few epochs only
    logging_strategy="epoch",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

**⚙️ Why a 10× smaller learning rate and only 3 epochs?**
* Large LR would **destroy pretrained knowledge** (catastrophic forgetting)
* Few epochs are enough for gentle adaptation

In [ ]:
trainer = Trainer(
    model=model,  # already trained model
    args=training_args_stage2,
    train_dataset=tokenized_data["train"],
    eval_dataset=tokenized_data["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

### 📊 What changed in metrics?

| Epoch | Val Loss |  Accuracy |       AUC |
| ----: | -------: | --------: | --------: |
|     1 |    0.326 |     0.887 |     0.957 |
|     2 |    0.312 | **0.902** | **0.961** |
|     3 |    0.334 |     0.902 |     0.961 |

**Jump from Stage 1 → Stage 2**

* Accuracy: **89% → 90.2%**
* AUC: **0.945 → 0.961**

A very noticeable improvement for unfreezing just **2 layers**.

---

**✅ What this proves**

* The frozen BERT already gave strong features
* But the top encoder layers were slightly **misaligned** with the phishing task
* Letting them adapt produced a **clear quality boost**

This is the sweet spot between:

* classifier-only tuning (fast but limited)
* full fine-tuning (slow, risk of overfitting)

👉 **Selective unfreezing gives the best trade-off.**

### 🧪 Stage 2 Evaluation on Test Set



In [26]:
# generate prediction
predictions = trainer.predict(tokenized_data["test"])

# get logits and labels
logits = predictions.predictions
labels = predictions.label_ids

# use created earlier compute_metrics
metrics = compute_metrics((logits, labels))
print(metrics)

{'Accuracy': np.float64(0.891), 'AUC': np.float64(0.97)}


**📈 What improved vs Stage 1**

| Stage                      |  Accuracy |       AUC |
| -------------------------- | --------: | --------: |
| Stage 1 (frozen encoder)   |     0.873 |     0.951 |
| Stage 2 (partial unfreeze) | **0.891** | **0.970** |

---

**💡 Key takeaway**

Fine-tuning the top encoder layers:

* improves **representation quality**, not just classification
* helps the model better adapt to **URL-specific patterns**
* significantly boosts **ranking ability (AUC)**

👉 This confirms that selective unfreezing of Transformer layers provides a strong performance gain without full retraining of the model.

### ☁️ Pushing the Model to Hugging Face Hub

After training, we upload the fine-tuned model to the Hugging Face Model Hub so it can be reused, shared, or deployed.

First we need to generate a `write` token at https://huggingface.co/settings/tokens and log in to the hub with this token.

In [27]:
# run this if you don't have HF_TOKEN in env variables
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
# push model to hub
trainer.push_to_hub()

https://huggingface.co/maksymstefanko/bert-phishing-classifier-stage2

## 🔍 Inference on New Instances (URL Classification)

Now we use our fine-tuned **BERT base uncased** model to classify unseen URLs as **Safe** or **Phishing**.

This is the final step: turning the trained model into a real prediction system.


In [29]:
# First, we check if CUDA (i.e. GPU) is available
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Use GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not availabel. Use CPU.")

# Transfer the model to the appropriate device (GPU or CPU)
model = model.to(device)


Use GPU: Tesla T4


In [37]:
# Tokenize input text
# input_text = "https://www.gogle.com"
input_text = "000mclogin.micloud-object-storage-xc-cos-static-web-hosting-qny.s3.us-east.cloud-object-storage.appdomain.cloud"
inputs = tokenizer(input_text, return_tensors="pt").to(device)

# turn off gradient calculations — we only make predictions
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

# Getting a text label from a numeric prediction
predicted_label = model.config.id2label[predictions.item()]
print(f"Predicted Label: {predicted_label}")

Predicted Label: phishing
